In [1]:
# ============================================================
# HYBRID STACK INFERENCE NOTEBOOK
# ML-Based Predictive Irrigation System for Paddy Cultivation
# Loads saved Phase 2 models and runs CWR predictions
# ============================================================

import numpy as np
import pandas as pd
import joblib
import os

print("=" * 60)
print("HYBRID STACK INFERENCE NOTEBOOK")
print("ML-Based Predictive Irrigation System")
print("=" * 60)
print("Libraries loaded successfully.")


In [2]:
# ============================================================
# Cell 2 - Load All Saved Models
# ============================================================

models_dir    = './Models'
retrained_dir = './Models/retrained'

# --- Load Retrained Family Champions ---
retrained_family1 = joblib.load(f"{retrained_dir}/retrained_family1_catboost.pkl")
retrained_family2 = joblib.load(f"{retrained_dir}/retrained_family2_svr.pkl")
retrained_family3 = joblib.load(f"{retrained_dir}/retrained_family3_mlp.pkl")

# --- Load Meta-Learner ---
meta_learner = joblib.load(f"{models_dir}/meta_learner_ridge.pkl")

print("=" * 60)
print("MODELS LOADED")
print("=" * 60)
print(f"Family 1 Champion : {type(retrained_family1).__name__}")
print(f"Family 2 Champion : {type(retrained_family2).__name__}")
print(f"Family 3 Champion : {type(retrained_family3.named_steps['mlp']).__name__} (Pipeline)")
print(f"Meta-Learner      : {type(meta_learner).__name__}")
print("\nAll models loaded successfully.")


In [3]:
# ============================================================
# Cell 3 - Define Hybrid Stack Prediction Function
# Input  : single sample or batch as dict or DataFrame
# Output : predicted CWR in mm
# ============================================================

FEATURE_COLUMNS = [
    'Water_Depth_cm',
    'Soil_Moisture_%',
    'Tank_Level_%',
    'ET_mm_day',
    'Rainfall_Predicted_mm',
    'Crop_Stage'
]

def predict_cwr(input_data):
    """
    Predict Crop Water Requirement (CWR) in mm using the hybrid stack.

    Parameters
    ----------
    input_data : dict or list of dicts or pd.DataFrame
        Input features. Must contain all 6 feature columns.

    Returns
    -------
    np.ndarray
        Predicted CWR values in mm.
    """
    # Convert input to DataFrame
    if isinstance(input_data, dict):
        X_input = pd.DataFrame([input_data], columns=FEATURE_COLUMNS)
    elif isinstance(input_data, list):
        X_input = pd.DataFrame(input_data, columns=FEATURE_COLUMNS)
    elif isinstance(input_data, pd.DataFrame):
        X_input = input_data[FEATURE_COLUMNS].copy()
    else:
        raise ValueError("input_data must be a dict, list of dicts, or DataFrame.")

    # Validate all required features are present
    missing = [f for f in FEATURE_COLUMNS if f not in X_input.columns]
    if missing:
        raise ValueError(f"Missing features: {missing}")

    # Base learner predictions
    meta_features = np.zeros((len(X_input), 3))
    meta_features[:, 0] = retrained_family1.predict(X_input)
    meta_features[:, 1] = retrained_family2.predict(X_input)
    meta_features[:, 2] = retrained_family3.predict(X_input)

    # Meta-learner final prediction
    cwr_predictions = meta_learner.predict(meta_features)

    return cwr_predictions


In [15]:
# ============================================================
# Cell 4 - Single Sample Prediction
# Hardcoded input representing one irrigation scenario
# ============================================================

single_sample = {
    'Water_Depth_cm':        1.0,    # current water depth in field
    'Soil_Moisture_%':      97.0,    # soil moisture percentage
    'Tank_Level_%':         70.0,    # water tank level percentage
    'ET_mm_day':             6.5,    # evapotranspiration mm/day
    'Rainfall_Predicted_mm': 1.0,    # predicted rainfall in mm
    'Crop_Stage':            5       # 1=vegetative, 2=reproductive,
                                     # 3=maturity, 4=ripening, 5=harvest
}

cwr_result = predict_cwr(single_sample)

print("=" * 60)
print("SINGLE SAMPLE PREDICTION")
print("=" * 60)
print("Input Features:")
for feature, value in single_sample.items():
    print(f"  {feature:<25} : {value}")
print("-" * 60)
print(f"Predicted CWR : {cwr_result[0]:.4f} mm")
print(f"Interpretation: Open gate to deliver {cwr_result[0]:.2f} mm of water")
